# Log Extractor — SSL / RoboCup (documento executável)

Extrai um log do `ssl-unification` (`<data>.log.gz`, container *recordio* + `RCLog.Log`) para tabelas
legíveis (parquet + `manifest.json`). **Toda a lógica vive no pacote** `dtssl.data.extract`
(porta fiel deste notebook, ver `docs/refactor_notes/etl.md`); este notebook só chama as funções.

Equivalente em linha de comando:

```bash
dtssl etl inspect logs/2026-05-18_19-2-15.log.gz
dtssl etl extract logs/2026-05-18_19-2-15.log.gz data/extracted/2026-05-18_19-2-15 [--csv]
```

**Saídas** em `OUTPUT_DIR/<log_name>/`: `ball`, `raw_ball`, `processed_robots`, `raw_robots`,
`commands`, `telemetry`, `referee` (`.parquet`; `.csv` opcional) + `manifest.json`
(`log_file`, `timeref_ns`, `tables`, `extraction`).

Requisitos: `pip install -e .` (numpy, pandas, pyarrow, protobuf). Não precisa de `protoc`
nem de `betterproto`: as classes protobuf estão versionadas em `dtssl.data._proto`.


## 1. Configuração

In [ ]:
from pathlib import Path

from dtssl.data.extract import dataset_name_from_log, extract_streams, inspect_log, write_dataset

# ------------------ CONFIG ------------------
PREFIX_PATH = Path('logs')
LOG_FILE    = '2026-05-18_19-2-15.log.gz'   # <-- troque pelo log atual (o sufixo .gz é um "misnomer": não é gzip)
OUTPUT_DIR  = Path('extracted')              # no repositório: data/extracted (DVC)
SAVE_CSV    = False                          # parquet é a fonte de verdade; csv só para inspeção
# --------------------------------------------

log_path = PREFIX_PATH / LOG_FILE
out_dir  = OUTPUT_DIR / dataset_name_from_log(LOG_FILE)
print(f'Log:   {log_path}')
print(f'Saída: {out_dir.resolve()}')


## 2. Inspeção rápida (contagens por stream, duração) — opcional

In [ ]:
info = inspect_log(log_path)
print(f"registros: {info['records']} (indecodificáveis {info['bad_records']}, truncado {info['truncated']})")
print(f"duração:   {info['duration_s']:.1f} s")
for name, n in info['counts'].items():
    print(f'{name:>18s}: {n:>8d}')


## 3. Extração + normalização de timestamps

`extract_streams` reproduz o notebook original: descarta `timestamp <= 1` ns, toma como referência o
menor primeiro `timestamp` entre `ball` e `raw_ball`, converte ns→s com
`((ts - timeref) % 1e13) / 1e9`, ordena por `timestamp` e acrescenta `velocity_norm`/`acceleration_norm`.
Unidades permanecem as do log (visão em **mm**, comandos em m/s e rad/s no frame de corpo).

In [ ]:
streams = extract_streams(log_path)
for name, df in streams.items():
    t = f"t=[{df['timestamp'].iloc[0]:.3f}, {df['timestamp'].iloc[-1]:.3f}] s" if len(df) else 'vazio'
    print(f'{name:>20s}: {len(df):>8d} linhas, {len(df.columns):>2d} colunas  {t}')
print(f"timeref (ns): {streams.meta['timeref_ns']:.0f}")


## 4. Salvamento (parquet + manifest.json)

In [ ]:
manifest_path = write_dataset(streams, out_dir, csv=SAVE_CSV)
print(f'Extração salva em: {out_dir.resolve()}')
for p in sorted(out_dir.iterdir()):
    print(f'  - {p.name:25s} ({p.stat().st_size/1024:8.1f} KB)')


## 5. Reler depois

```python
import pandas as pd
from dtssl.data.extract import read_dataset, read_manifest
ball = pd.read_parquet('extracted/<log_name>/ball.parquet')
dfs = read_dataset('extracted/<log_name>')      # dict com os sete streams
manifest = read_manifest('extracted/<log_name>')
```

Próximo passo: `02_splitByLabels.ipynb` (ou `dtssl etl split <dataset_dir> [labels.csv]`).